# Demo usage of the common interface for generating embeddings

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add project root to sys.path to allow imports
sys.path.append("..")

from helpers.data_loaders import load_movielens_data, load_steam_data
from helpers.evaluation import leave_one_out_split, calculate_metrics, print_metrics
from src.models import (
    LightGCNGenerator,
    Node2VecGenerator,
    CleoraGenerator,
    MatrixFactorizationGenerator,
    NCFGenerator
)


## 1. Data preprocessing

In [56]:
# Load MovieLens data
movies_df, ratings_df = load_movielens_data("datasets/movies/movies.csv", "datasets/movies/ratings.csv")
movielens_interactions = pd.DataFrame({
    'user_id': 'ml_user_' + ratings_df['userId'].astype(str),
    'item_id': 'ml_item_' + ratings_df['movieId'].astype(str),
    'rating': ratings_df['rating'],
    'timestamp': ratings_df['timestamp']
}).drop_duplicates(subset=['user_id', 'item_id'])
 
# Preprocess MovieLens: Filter for high ratings to treat as positive interactions
movielens_interactions = movielens_interactions[movielens_interactions['rating'] >= 4.0].copy()

# Load Steam data
reviews_df_steam, items_df_steam = load_steam_data(
    "datasets/steam/formatted_user_reviews.json",
    "datasets/steam/formatted_steam_games.json"
)
steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str),
    'rating': 1.0, # Steam reviews are implicit feedback, use 1.0 for positive
    'timestamp': 0 # Default timestamp for Steam reviews (no date available)
}).drop_duplicates(subset=['user_id', 'item_id'])

# Concatenate and drop duplicates
all_interactions = pd.concat([movielens_interactions[['user_id', 'item_id', 'rating', 'timestamp']],
                              steam_interactions[['user_id', 'item_id', 'rating', 'timestamp']]])
all_interactions = all_interactions.drop_duplicates(subset=['user_id', 'item_id']).reset_index(drop=True)

# Sample 100,000 interactions if the dataset is larger
if len(all_interactions) > 500000:
    interactions_df = all_interactions.sample(n=500000, random_state=42).reset_index(drop=True)
else:
    interactions_df = all_interactions

print(f"All Unique users: {all_interactions['user_id'].nunique()}")
print(f"All Unique items: {all_interactions['item_id'].nunique()}")

print(f"Using {len(interactions_df)} interactions for demonstration.")
print(f"Unique users: {interactions_df['user_id'].nunique()}")
print(f"Unique items: {interactions_df['item_id'].nunique()}")

# Split Data (Leave-One-Out)
train_df, test_df = leave_one_out_split(interactions_df, time_col='timestamp')
print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")


All Unique users: 184419
All Unique items: 43660
Using 500000 interactions for demonstration.
Unique users: 123229
Unique items: 15558
Train size: 376771
Test size: 123229


## 2. Define Models

In [59]:
models = [
    ("LightGCN", LightGCNGenerator(epochs=5, batch_size=128)),
    ("Node2Vec", Node2VecGenerator(epochs=5, batch_size=128)),
    ("MatrixFactorization", MatrixFactorizationGenerator(epochs=5, batch_size=128)),
    ("NCF", NCFGenerator(epochs=5, batch_size=128)),
    ("Cleora", CleoraGenerator(num_walks=2))
]

## 3. Train and Evaluate

In [63]:
results_dir = 'models'
os.makedirs(results_dir, exist_ok=True)

for name, model in models:
    print(f"\n--- Testing {name} ---")
    try:
        # Determine filename
        filename = f"partial_{name.lower()}_model.pth"
        
        load_path = os.path.join(results_dir, filename)
        loaded = False
        
        if os.path.exists(load_path):
            print(f"Found checkpoint at {load_path}. Attempting to load...")
            try:
                model.load(load_path)
                print("Model loaded successfully.")
                loaded = True
            except Exception as e:
                print(f"Failed to load {name} (will retrain): {e}")
        else:
            print(f"No checkpoint found at {load_path}. Will train from scratch.")
        
        if not loaded:
            print(f"Training {name}...")
            model.fit(train_df, user_col='user_id', item_col='item_id', rating_col='rating')
            print(f"Saving model to {load_path}...")
            model.save(load_path)
        
        print(f"Generating embeddings...")
        embeddings = model.get_embeddings()
        
        print(f"Generated {len(embeddings)} embeddings.")
        # Print shape of a sample embedding
        if embeddings:
            sample_key = list(embeddings.keys())[0]
            print(f"Shape of embedding for '{sample_key}': {embeddings[sample_key].shape}")

        # Evaluate
        print(f"Evaluating {name}...")
        metrics = calculate_metrics(model, train_df, test_df)
        print_metrics(metrics)
        
    except ImportError as e:
        print(f"Skipping {name} due to missing dependency: {e}")
    except Exception as e:
        print(f"An error occurred with {name}: {e}")
        import traceback
        traceback.print_exc()



--- Testing LightGCN ---
Found checkpoint at models\partial_lightgcn_model.pth. Attempting to load...
Model loaded successfully.
Generating embeddings...
Generated 98501 embeddings.
Shape of embedding for 'ml_user_1000': (64,)
Evaluating LightGCN...


Evaluating:   0%|          | 0/123229 [00:00<?, ?it/s]


Evaluation Results:
------------------------------
Precision@20: 0.0030
Precision@50: 0.0026
Precision@100: 0.0022
Precision@1000: 0.0007
HitRate@20: 0.0595
HitRate@50: 0.1290
HitRate@100: 0.2210
HitRate@1000: 0.7162
NDCG@20: 0.0216
NDCG@50: 0.0352
NDCG@100: 0.0501
NDCG@1000: 0.1098
MAP@20: 0.0114
MAP@50: 0.0136
MAP@100: 0.0149
MAP@1000: 0.0167
Diversity@20: 0.1342
Diversity@50: 0.1489
Diversity@100: 0.1667
Diversity@1000: 0.2559
Novelty@20: 8.8381
Novelty@50: 9.1411
Novelty@100: 9.4603
Novelty@1000: 11.5608
Serendipity@20: 0.1377
Serendipity@50: 0.1341
Serendipity@100: 0.1378
Serendipity@1000: 0.2104
AveragePopularity@20: 940.9816
AveragePopularity@50: 780.1606
AveragePopularity@100: 643.8084
AveragePopularity@1000: 215.1183
AUC (Global): 0.9119
AUC (Sampled): 0.9119
MeanRank: 1231.2251
MRR: 0.0168
------------------------------

--- Testing Node2Vec ---
Found checkpoint at models\partial_node2vec_model.pth. Attempting to load...
Model loaded successfully.
Generating embeddings...
Ge

Evaluating:   0%|          | 0/123229 [00:00<?, ?it/s]


Evaluation Results:
------------------------------
Precision@20: 0.0024
Precision@50: 0.0021
Precision@100: 0.0018
Precision@1000: 0.0006
HitRate@20: 0.0479
HitRate@50: 0.1057
HitRate@100: 0.1839
HitRate@1000: 0.6145
NDCG@20: 0.0170
NDCG@50: 0.0283
NDCG@100: 0.0410
NDCG@1000: 0.0928
MAP@20: 0.0088
MAP@50: 0.0105
MAP@100: 0.0116
MAP@1000: 0.0132
Diversity@20: 0.1886
Diversity@50: 0.1836
Diversity@100: 0.1906
Diversity@1000: 0.4546
Novelty@20: 9.8370
Novelty@50: 9.9987
Novelty@100: 10.2630
Novelty@1000: 12.4778
Serendipity@20: 0.2145
Serendipity@50: 0.2025
Serendipity@100: 0.2009
Serendipity@1000: 0.3419
AveragePopularity@20: 858.8707
AveragePopularity@50: 726.7295
AveragePopularity@100: 605.4076
AveragePopularity@1000: 202.2543
AUC (Global): 0.8379
AUC (Sampled): 0.8380
MeanRank: 2263.4495
MRR: 0.0133
------------------------------

--- Testing MatrixFactorization ---
Found checkpoint at models\partial_matrixfactorization_model.pth. Attempting to load...
Failed to load MatrixFactorizat

...

Saving model to models\partial_ncf_model.pth...
Generating embeddings...
Generated 98501 embeddings.
Shape of embedding for 'ml_user_1000': (64,)
Evaluating NCF...


Evaluating:   0%|          | 0/123229 [00:00<?, ?it/s]


Evaluation Results:
------------------------------
Precision@20: 0.0004
Precision@50: 0.0004
Precision@100: 0.0003
Precision@1000: 0.0002
HitRate@20: 0.0084
HitRate@50: 0.0177
HitRate@100: 0.0317
HitRate@1000: 0.1622
NDCG@20: 0.0030
NDCG@50: 0.0048
NDCG@100: 0.0071
NDCG@1000: 0.0224
MAP@20: 0.0016
MAP@50: 0.0018
MAP@100: 0.0020
MAP@1000: 0.0024
Diversity@20: 0.6707
Diversity@50: 0.6796
Diversity@100: 0.6813
Diversity@1000: 0.6251
Novelty@20: 13.3420
Novelty@50: 13.8015
Novelty@100: 14.2065
Novelty@1000: 15.8433
Serendipity@20: 0.9799
Serendipity@50: 1.0054
Serendipity@100: 1.0272
Serendipity@1000: 1.1123
AveragePopularity@20: 177.0557
AveragePopularity@50: 155.3105
AveragePopularity@100: 136.5975
AveragePopularity@1000: 65.4420
AUC (Global): 0.5284
AUC (Sampled): 0.5284
MeanRank: 6584.8711
MRR: 0.0026
------------------------------

--- Testing Cleora ---
Found checkpoint at models\partial_cleora_model.pth. Attempting to load...
Model loaded successfully.
Generating embeddings...
Gene

Evaluating:   0%|          | 0/123229 [00:00<?, ?it/s]


Evaluation Results:
------------------------------
Precision@20: 0.0001
Precision@50: 0.0001
Precision@100: 0.0001
Precision@1000: 0.0001
HitRate@20: 0.0014
HitRate@50: 0.0033
HitRate@100: 0.0072
HitRate@1000: 0.0692
NDCG@20: 0.0005
NDCG@50: 0.0009
NDCG@100: 0.0015
NDCG@1000: 0.0085
MAP@20: 0.0002
MAP@50: 0.0003
MAP@100: 0.0003
MAP@1000: 0.0005
Diversity@20: 0.8553
Diversity@50: 0.8833
Diversity@100: 0.8975
Diversity@1000: 0.9348
Novelty@20: 16.1614
Novelty@50: 16.1988
Novelty@100: 16.2140
Novelty@1000: 16.2690
Serendipity@20: 0.9586
Serendipity@50: 0.9573
Serendipity@100: 0.9559
Serendipity@1000: 0.9527
AveragePopularity@20: 27.8475
AveragePopularity@50: 27.5440
AveragePopularity@100: 27.3009
AveragePopularity@1000: 26.7631
AUC (Global): 0.4973
AUC (Sampled): 0.4973
MeanRank: 7019.0938
MRR: 0.0007
------------------------------
